# Analyzing the distance covered and anomalities

In [1]:
import duckdb

In [2]:
df = duckdb.read_parquet('../raw/*/*.parquet')
df

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ airport_fee │
│  int64   │      timestamp       │       timestamp       │     double      │    double     │   double   │      varchar       │    int64     │    int64     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

## Filtering out extreme outliers (Distance)

In [8]:
zero = duckdb.sql("""
                    SELECT
                        trip_distance,
                        fare_amount,
                        total_amount
                    FROM 
                        df
                    WHERE
                        trip_distance ==  0
                    """).df()
zero

,trip_distance,fare_amount,total_amount
0,0.0,2.50,6.30
1,0.0,2.50,6.30
2,0.0,2.50,6.30
3,0.0,2.50,6.30
4,0.0,10.00,15.36
...,...,...,...
3526774,0.0,22.81,27.95
3526775,0.0,50.66,55.41
3526776,0.0,27.50,43.12
3526777,0.0,9.19,13.94


**There are over `3.5M` rows where distance travelled is 0:**
* For some of them faires are charged suggesting it might be a cancelled ride
* It could be useful for revenue analysis

In [ ]:
td_l_0 = duckdb.sql("""
                    SELECT
                        trip_distance,
                        fare_amount,
                        total_amount
                    FROM 
                        df
                    WHERE
                        trip_distance > 500
                    """).df()
td_l_0

,trip_distance,fare_amount,total_amount
0,577.50,52.00,71.13
1,622.00,52.00,75.70
2,572.60,52.00,61.85
3,616.80,52.00,75.70
4,651.00,52.00,71.85
...,...,...,...
5147,95310.01,86.69,107.39
5148,92140.64,46.82,48.32
5149,139278.22,24.75,29.50
5150,221055.79,29.93,38.14


In [ ]:
td_l_0.describe()

,trip_distance,fare_amount,total_amount
count,5152.000000,5152.000000,5152.000000
mean,69017.704606,32.847125,39.677937
std,61853.257809,645.163694,645.268910
min,500.640000,-633.110000,-636.360000
25%,19448.175000,12.627500,17.587500
50%,58141.365000,19.895000,25.195000
75%,99157.192500,31.440000,38.177500
max,398608.620000,46263.880000,46269.440000


**From the above analysis it is clear that there are some anomalities in the trip distance column**

*For example:*
* Earth's Circumference: `24,901.461 mi`
* The maximum trip distance recorded is:`398608.62 mi`
    * There for the maximum noted trip distance is `~16 times` earth's circumference

**There are about 5152 rows which contain distances greater than 500mi** 
* Removing those data to get a cleaner dataset

### Filtering out rows where trip_distance is 0mi and greater than 500 

In [9]:
df1 = duckdb.sql("""
                    SELECT
                        *
                    FROM 
                        df
                    WHERE
                        trip_distance < 500 
                    AND
                       trip_distance <> 0 
                    """)
df1

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ airport_fee │
│  int64   │      timestamp       │       timestamp       │     double      │    double     │   double   │      varchar       │    int64     │    int64     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

In [ ]:
duckdb.sql("""
            SELECT
                COUNT (*)
            FROM
                df1
            WHERE 
                trip_distance > 100
            """)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         2164 │
└──────────────┘

**There are only 2164 rows with miles greater than 100**

* As 75% of the data falls with 3.5 mi range it is highly likely that the trip distance of more than 100mi are anomalities

### Filtering out rows where trip_distance is greater than 100

In [23]:
df1 = duckdb.sql("""
                    SELECT
                        *
                    FROM 
                        df
                    WHERE
                        trip_distance < 100
                    """)
df1

┌──────────┬──────────────────────┬───────────────────────┬─────────────────┬───────────────┬────────────┬────────────────────┬──────────────┬──────────────┬──────────────┬─────────────┬────────┬─────────┬────────────┬──────────────┬───────────────────────┬──────────────┬──────────────────────┬─────────────┐
│ VendorID │ tpep_pickup_datetime │ tpep_dropoff_datetime │ passenger_count │ trip_distance │ RatecodeID │ store_and_fwd_flag │ PULocationID │ DOLocationID │ payment_type │ fare_amount │ extra  │ mta_tax │ tip_amount │ tolls_amount │ improvement_surcharge │ total_amount │ congestion_surcharge │ airport_fee │
│  int64   │      timestamp       │       timestamp       │     double      │    double     │   double   │      varchar       │    int64     │    int64     │    int64     │   double    │ double │ double  │   double   │    double    │        double         │    double    │        double        │   double    │
├──────────┼──────────────────────┼───────────────────────┼───────────

In [24]:
trip_tw = duckdb.sql("""
                SELECT 
                    trip_distance
                FROM df1
                ORDER BY trip_distance DESC
                """).df()

trip_tw

,trip_distance
0,99.92
1,99.91
2,99.91
3,99.90
4,99.89
...,...
167851325,0.00
167851326,0.00
167851327,0.00
167851328,0.00


In [25]:
trip_tw.describe()

,trip_distance
count,1.678513e+08
mean,3.427596e+00
std,4.423333e+00
min,0.000000e+00
25%,1.050000e+00
50%,1.810000e+00
75%,3.500000e+00
max,9.992000e+01


## Extracting Trip Distance Time

In [27]:
# trip_tw = duckdb.sql("""
#                 SELECT 
#                     trip_distance,
#                     EXTRACT(EPOCH FROM (
#                         tpep_dropoff_datetime - tpep_pickup_datetime
#                         )) / 60 AS trip_duration_minutes,
#                     fare_amount,
#                     total_amount
#                 FROM df1
#                 ORDER BY trip_distance DESC
#                 """).df()

trip_tw

,trip_distance
0,99.92
1,99.91
2,99.91
3,99.90
4,99.89
...,...
167851325,0.00
167851326,0.00
167851327,0.00
167851328,0.00
